In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\LENOVO\Desktop\MLOPS-Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
221,I have been watching movies from i think last ...,negative
686,"Firstly,I must admit that it isn't a good movi...",negative
101,"What can I say? I think I have to write ""Spoil...",negative
138,Footlight Parade is among the best of the 1930...,positive
468,I can say without a shadow of a doubt that Goi...,negative


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21668\3798096103.py:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  text = re.sub('\s+', ' ', text).strip()


In [5]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...


True

In [6]:
df = normalize_text(df)
df.head()

,review,sentiment
221,watching movie think last year must say never ...,negative
686,firstly must admit good movie would never watc...,negative
101,say think write spoiler alert reveal used f wo...,negative
138,footlight parade among best musical comedy ext...,positive
468,say without shadow doubt going overboard singl...,negative


In [7]:
df['sentiment'].value_counts()

sentiment
negative    272
positive    228
Name: count, dtype: int64

In [8]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
221,watching movie think last year must say never ...,0
686,firstly must admit good movie would never watc...,0
101,say think write spoiler alert reveal used f wo...,0
138,footlight parade among best musical comedy ext...,1
468,say without shadow doubt going overboard singl...,0


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/sup-neupane/MLOPS-Project.mlflow')
dagshub.init(repo_owner='sup-neupane', repo_name='MLOPS-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as sup-neupane

Initialized MLflow to track repo "sup-neupane/MLOPS-Project"

Repository sup-neupane/MLOPS-Project initialized!

2026/11/24 12:32:47 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2499224e1f2b4cc2842a0862fdefa181', creation_time=1795502868730, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1795502868730, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [14]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-11-24 12:36:07,079 - INFO - Starting MLflow run...
2026-11-24 12:36:09,405 - INFO - Logging preprocessing parameters...
2026-11-24 12:36:12,092 - INFO - Initializing Logistic Regression model...
2026-11-24 12:36:12,093 - INFO - Fitting the model...
2026-11-24 12:36:12,224 - INFO - Model training complete.
2026-11-24 12:36:12,226 - INFO - Logging model parameters...
2026-11-24 12:36:12,652 - INFO - Making predictions...
2026-11-24 12:36:12,655 - INFO - Calculating evaluation metrics...
2026-11-24 12:36:12,680 - INFO - Logging evaluation metrics...
2026-11-24 12:36:14,196 - INFO - Saving and logging the model...
2026/11/24 12:36:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-11-24 12:36:40,930 - INFO - Model training and logging completed in 31.52 seconds.
2026-11-24 12:36:40,932 - INFO - Accuracy: 0.68
2026-11-24 12:36:40,933 - INFO - Precision: 0.7058823529411765
2026-11-24 12:36:40,934 - INFO - Recall: 0.5901639344262295
2026-11-24 

🏃 View run clumsy-zebra-565 at: https://dagshub.com/sup-neupane/MLOPS-Project.mlflow/#/experiments/0/runs/c1d68fbfd45246d887a7aaea2c1fb8bb
🧪 View experiment at: https://dagshub.com/sup-neupane/MLOPS-Project.mlflow/#/experiments/0
